In [ ]:
!pip install apted lxml
!pip install -U transformers==4.41.2 accelerate peft bitsandbytes datasets
!pip install -U transformers accelerate peft bitsandbytes

  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)
  Using cached huggingface_hub-1.7.1-py3-none-any.whl.metadata (13 kB)
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
  Using cached hf_xet-1.4.2-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (4.9 kB)
Using cached transformers-5.3.0-py3-none-any.whl (10.7 MB)
Using cached huggingface_hub-1.7.1-py3-none-any.whl (616 kB)
Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
Using cached hf_xet-1.4.2-cp37-abi3-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (4.2 MB)
  Attempting uninstall: hf-xet
    Found existing installation: hf-xet 1.3.2
    Uninstalling hf-xet-1.3.2:
      Successfully uninstalled hf-xet-1.3.2
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.

In [ ]:
#Load FintuneModel
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel

base_model_id = "Qwen/Qwen2.5-VL-3B-Instruct"
adapter_path = "/content/drive/MyDrive/DocumentDataSet/FINAL_MODEL_2K/"

# 4-bit config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

# Load base model first
model = AutoModelForImageTextToText.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Load LoRA adapter
model = PeftModel.from_pretrained(model, adapter_path)

processor = AutoProcessor.from_pretrained(
    base_model_id,
    trust_remote_code=True
)

processor.tokenizer.padding_side="left"
processor.tokenizer.pad_token=processor.tokenizer.eos_token
model.eval()

print("✅ Base model + LoRA adapter loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Base model + LoRA adapter loaded successfully.


In [ ]:
# ==========================================================
# BATCHED RESUMABLE VALIDATION (GPU OPTIMIZED)
# ==========================================================

import os
import json
import torch
from PIL import Image
from apted import APTED, Config
from lxml import etree
from io import StringIO

# ---------------- SETTINGS ----------------
MAX_IMAGES = 1000
MAX_TOKENS = 768
BATCH_SIZE = 6   # 🔥 Try 4 first. If GPU stable → increase to 6

val_folder = "/content/drive/MyDrive/DocumentDataSet/val/"
json_path = "/content/drive/MyDrive/DocumentDataSet/PubTabNet_2.0.0.jsonl"

progress_file = "/content/drive/MyDrive/DocumentDataSet/val_1000_progress_2k.json"
log_file = "/content/drive/MyDrive/DocumentDataSet/val_1000_logs_2k.json"

# ---------------- LOAD GT ----------------
print("Loading ground truth...")
gt_dict = {}

with open(json_path, "r", encoding="utf-8") as f:
    for line in f:
        entry = json.loads(line)
        gt_dict[entry["filename"]] = "".join(entry["html"]["structure"]["tokens"])

print("GT loaded.\n")

# ---------------- TEDS ----------------
class TableTreeConfig(Config):
    def children(self, node):
        return list(node)

    def rename(self, node1, node2):
        return 0 if node1.tag == node2.tag else 1

def html_to_tree(html_string):
    parser = etree.HTMLParser(remove_blank_text=True)
    tree = etree.parse(StringIO(html_string), parser)
    return tree.getroot()

def compute_teds(gt_html, pred_html):
    gt_tree = html_to_tree(gt_html)
    pred_tree = html_to_tree(pred_html)
    apted = APTED(gt_tree, pred_tree, TableTreeConfig())
    edit_distance = apted.compute_edit_distance()
    max_nodes = max(len(list(gt_tree.iter())), len(list(pred_tree.iter())))
    return 1 - (edit_distance / max_nodes)

# ---------------- LOAD PROGRESS ----------------
if os.path.exists(progress_file):
    with open(progress_file, "r") as f:
        saved = json.load(f)
    processed_images = set(saved["processed"])
    teds_scores = saved["scores"]
    print(f"Resuming from {len(processed_images)} images\n")
else:
    processed_images = set()
    teds_scores = []

if os.path.exists(log_file):
    with open(log_file, "r") as f:
        detailed_logs = json.load(f)
else:
    detailed_logs = []

# ---------------- IMAGE LIST ----------------
image_list = sorted(os.listdir(val_folder))[:MAX_IMAGES]
image_list = [img for img in image_list if img in gt_dict and img not in processed_images]

print(f"Remaining images: {len(image_list)}\n")

# ---------------- BATCH LOOP ----------------
for i in range(0, len(image_list), BATCH_SIZE):

    batch_images = image_list[i:i+BATCH_SIZE]
    images = []
    texts = []

    for image_name in batch_images:
        image_path = os.path.join(val_folder, image_name)
        image = Image.open(image_path).convert("RGB")
        images.append(image)

        messages = [{
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": "Extract the full table structure in HTML format."}
            ]
        }]

        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        texts.append(text)

    inputs = processor(
        text=texts,
        images=images,
        return_tensors="pt",
        padding=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_TOKENS,
            do_sample=False,
            num_beams=1
        )

    decoded = processor.batch_decode(outputs, skip_special_tokens=True)

    # -------- PROCESS EACH OUTPUT --------
    for image_name, output_text in zip(batch_images, decoded):

        assistant_output = output_text.split("assistant")[-1].strip()
        gt_html = gt_dict[image_name]

        try:
            score = compute_teds(gt_html, assistant_output)
            teds_scores.append(score)
            processed_images.add(image_name)

            detailed_logs.append({
                "image": image_name,
                "teds": score
            })

        except:
            continue

    # -------- SAVE PROGRESS --------
    if len(processed_images) % 20 == 0:
        with open(progress_file, "w") as f:
            json.dump({
                "processed": list(processed_images),
                "scores": teds_scores
            }, f)

        with open(log_file, "w") as f:
            json.dump(detailed_logs, f)

        current_avg = sum(teds_scores) / len(teds_scores)
        print(f"Processed {len(processed_images)}/{MAX_IMAGES} | Avg TEDS: {current_avg:.4f}")

# ---------------- FINAL SAVE ----------------
with open(progress_file, "w") as f:
    json.dump({
        "processed": list(processed_images),
        "scores": teds_scores
    }, f)

with open(log_file, "w") as f:
    json.dump(detailed_logs, f)

print("\n" + "="*60)
if len(teds_scores) > 0:
    final_avg = sum(teds_scores) / len(teds_scores)
    print(f"🔥 FINAL AVG TEDS: {final_avg:.4f}")
else:
    print("No valid scores computed.")
print("="*60)

Loading ground truth...
GT loaded.

Resuming from 800 images

Remaining images: 200



The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
/usr/local/lib/python3.12/dist-packages/apted/node_indexer.py:331: FutureWarning: Truth-testing of elements was a source of confusion and will always return True in future versions. Use specific 'len(elem)' or 'elem is not None' test instead.
  return bool(self.node)


Processed 860/1000 | Avg TEDS: 0.6649
Processed 920/1000 | Avg TEDS: 0.6631
Processed 980/1000 | Avg TEDS: 0.6679
Processed 1000/1000 | Avg TEDS: 0.6699

🔥 FINAL AVG TEDS: 0.6699
